# モデル推論(LightGBM)

学習済みモデルを利用してテストデータの予測を実施するフェーズです。

## 実施内容
* テストデータ読み込み
* 学習済モデル読み込み
* 学習時と同一の前処理適用
* 学習済みモデルによる予測
* 予測結果のCSV出力

## 1. ライブラリインポート

In [40]:
# 標準ライブラリ
import logging
import time
import warnings
from datetime import datetime
from functools import wraps
from pathlib import Path
from typing import Dict, Any, Callable, List, Tuple
from dataclasses import dataclass



# サードパーティライブラリ
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm.auto import tqdm
import onnxruntime as ort

# 自作モジュール
from utils_copy import(
    timing_decorator,
    get_latest_output_dir,
    
    

)
# ロガー設定
# logging_config.pyの設計方針を参考にloggerを初期化
# INFO以上(INFO, WARNING, ERROR, CRITICAL)を出力
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

warnings.filterwarnings("default")

logger.info("ライブラリのインポート完了")

2026-06-11 07:43:11,401 - INFO - ライブラリのインポート完了


In [37]:
# 共通化
# モジュール化候補
# 実行時間を記録

def timing_decorator(func: Callable) -> Callable:
    """関数の実行時間を計測してログ出力するデコレータ。"""

    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time: float = time.time()
        try:
            result = func(*args, **kwargs)
        except Exception:
            logger.exception(f"{func.__name__} でエラーが発生しました")
            raise
        elapsed_time: float = time.time() - start_time
        logger.info(f"{func.__name__} 実行時間: {elapsed_time:.2f}秒")
        return result
    return wrapper

## 2. パス設定
* 入力・出力ファイルのパスを定義する
* 03で作成したタイムスタンプ付き出力ディレクトリを参照する
* モデルを読み込み、推論結果を保存する

In [38]:
# 共通化
# 最新のoutputディレクトリを取得
@timing_decorator
def get_latest_output_dir(base_dir: str = "../output") -> Tuple[Path, str]:
    base = Path(base_dir)
    dirs = [d for d in base.iterdir() if d.is_dir()]

    if not dirs:
        raise FileNotFoundError("No output directories found.")

    latest = max(dirs, key=lambda d: d.name)
    timestamp = latest.name
    return latest, timestamp
# @timing_decorator

In [45]:
# 出力ルートディレクトリ（ローカル / Blobどちらも対応）

# ディレクトリ
# OUTPUT_DIR, OUTPUT_TIMESTAMP = get_latest_output_dir()

OUTPUT_DIR= "../output/2606101938"
OUTPUT_TIMESTAMP = "2606101938"

MODEL_DIR: str = f"{OUTPUT_DIR}/model"
ARTIFACT_DIR: str = f"{OUTPUT_DIR}/artifacts"


# 入力
TEST_DATA_PATH: str = "../data/test.csv"
# モデルファイル（推論用）
MODEL_BASELINE_FILE: str = f"{MODEL_DIR}/model_baseline_{OUTPUT_TIMESTAMP}.onnx"
MODEL_TUNED_FILE: str = f"{MODEL_DIR}/model_tuned_{OUTPUT_TIMESTAMP}.onnx"


# 出力
# 予測結果
PREDICTION_BASELINE_FILE: str = (
    f"{ARTIFACT_DIR}/prediction_baseline_{OUTPUT_TIMESTAMP}.csv"
)
PREDICTION_TUNED_FILE: str = (
    f"{ARTIFACT_DIR}/prediction_tuned_{OUTPUT_TIMESTAMP}.csv"
)

## 3. 変数定義
* 推論に使用する変数を定義

In [41]:
@dataclass
class FeatureSchema:
    target: str
    num_cols: List[str]
    cat_cols: List[str]
    
schema = FeatureSchema(
    target="label",
    num_cols=['id', 'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'],
    cat_cols= ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']   
)

In [ ]:
# 予測値の丸め桁数
ROUND_DIGITS = 3

# 区切り線の長さ
SEPARATOR_LENGTH = 70

# データ読み込み時のチャンクサイズ
CHUNK_SIZE = 1000

## 4. テストデータ読み込み

In [42]:
# ==共通化== →utils.py
# データの読み込み（チャンク化して進捗バー表示）
@timing_decorator
def load_data_pd(path: str, chunk_size: int) -> pd.DataFrame:
    chunks = []
    for chunk in tqdm(pd.read_csv(path, chunksize=chunk_size)):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

In [43]:
# テストデータの読み込み
X_test = load_data_pd(TEST_DATA_PATH, CHUNK_SIZE)

0it [00:00, ?it/s]

2026-06-11 07:44:05,550 - INFO - load_data_pd 実行時間: 0.07秒


## 5. 学習済モデル読み込み

In [46]:
# baseline
model_baseline = ort.InferenceSession(MODEL_BASELINE_FILE)
# tuned
# model_tuned = ort.InferenceSession(MODEL_TUNED_FILE)

## 6. 前処理

In [50]:
from utils_copy import to_categorical
X_test = to_categorical(X_test, schema.cat_cols)

In [48]:
input_name = model_baseline.get_inputs()[0].name

In [49]:
print(input_name)

input


## 7. 推論実行

In [34]:
def predict_onnx(model, X_np):

    input_name = model.get_inputs()[0].name

    preds = model.run(
        None,
        {input_name: X_np}
    )[0]

    return preds

In [35]:
# baseline
X_np = preprocess_for_onnx(X_test)

y_pred_proba = predict_onnx(model_baseline, X_np)
# prediction_baseline = predict(model_baseline, X_test_pd, ROUND_DIGITS)
# tuned
# prediction_tuned = predict(model_tuned, X_test_pd, ROUND_DIGITS)

InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: input for the following indices
 index: 1 Got: 8 Expected: 17
 Please fix either the inputs/outputs or the model.

## 8. 推論結果の保存

In [ ]:
# 予測結果をCSVへ保存
# baseline
prediction_baseline.to_csv(PREDICTION_BASELINE_FILE, index=False)
# tuned
prediction_tuned.to_csv(PREDICTION_TUNED_FILE, index=False)

# サンプル表示
logger.info(f"\n{'='*SEPARATOR_LENGTH}\n予測結果サンプル（先頭10件）\n{'='*SEPARATOR_LENGTH}")
logger.info(f"\nbaseline:\n{prediction_baseline.head(10)}")
logger.info(f"\ntuned:\n{prediction_tuned.head(10)}")